ETL - JUNÇÃO DAS TABELAS PARA ANÁLISE

In [1]:
import pandas as pd

clientes = pd.read_csv('Datasets/crm_clientes.csv',parse_dates=['data_cadastro'])
contatos = pd.read_csv('Datasets/crm_contatos.csv',parse_dates=['data_contato'])
propostas = pd.read_csv('Datasets/crm_propostas.csv',parse_dates=['data_envio'])

# display(clientes.sample(5))
# display(contatos.sample(5))
# display(propostas.sample(5))

contatos_agg = contatos.groupby(by='cliente_id').agg({'contato_id': 'count', 'tempo_resposta_horas': 'mean', 'data_contato': 'max'})
propostas_agg = propostas.groupby(by='cliente_id').agg({'proposta_id': 'count','valor_proposta': 'mean'})

clientes_contatos = pd.merge(left=clientes,right=contatos_agg,how='left',on='cliente_id',suffixes=('_clientes','_contatos'))

df_group = clientes_contatos.merge(right=propostas_agg,how='left',on='cliente_id',suffixes=('_clientes_contatos','propostas'))

df_group.sample(5)


target = (
    propostas
    .assign(comprou=lambda df: df['status_proposta'] == 'Ganha')
    .groupby('cliente_id')['comprou']
    .max()
    .reset_index()
)



df_group['valor_proposta'] = df_group['valor_proposta'].round(decimals=2)

df_group['tempo_resposta_horas'] = df_group['tempo_resposta_horas'].round(decimals=0)

df_group = df_group.fillna(0)

df_group = df_group.merge(right=target,how='left', on='cliente_id')

df_group = df_group.fillna(False)

df_group = df_group[['cliente_id','tipo_cliente','proposta_id','contato_id','tempo_resposta_horas','valor_proposta','data_contato','comprou']]


df_group.columns = ['cliente_id','tipo_cliente','qtde_proposta','qtde_contato', 'media_tempo_proposta','media_valor_proposta','ultimo_contato','comprou']

x = df_group[['cliente_id','qtde_proposta','qtde_contato', 'media_tempo_proposta','media_valor_proposta']]

# x = pd.concat([df_group[['qtde_proposta','qtde_contato','media_tempo_proposta','media_valor_proposta']], pd.get_dummies(df_group['tipo_cliente'], prefix='tipo', drop_first=True)], axis=1)


y = df_group['comprou']

# x = x.fillna(0)



C:\Users\Igor\AppData\Local\Temp\ipykernel_14588\3618612225.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_group = df_group.fillna(False)


SERAPAÇÃO DO DATASET DE TREINAMENTO E DATASET DE TESTE

In [2]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=16,stratify=y)




MODELO DE REGRESSÃO LOGISTICA

In [50]:
from sklearn.linear_model import LogisticRegression

cliente_id_merge = x_test['cliente_id'].reset_index().drop('index',axis=1)

logreg = LogisticRegression(random_state=16)

# fit

logreg.fit(x_train.drop('cliente_id', axis=1), y_train)

y_pred = logreg.predict(x_test.drop('cliente_id', axis=1))
y_pred_proba = logreg.predict_proba(x_test.drop('cliente_id', axis=1))
# display(y_pred_proba)



teste = pd.DataFrame(x_test.drop('cliente_id', axis=1)).reset_index().drop('index',axis=1)

probs = pd.DataFrame(y_pred_proba)

# display(probs)



final = pd.concat([teste,probs, cliente_id_merge],axis = 1)

final.rename(columns={0:'%_não_comprar', 1: '%_comprar'},inplace=True)



final['%_não_comprar'] = (final['%_não_comprar']*100).round(decimals=2)
final['%_comprar'] = (final['%_comprar']*100).round(decimals=2)

final = final.sort_values(by='%_comprar',ascending=False)

final = final[final['%_comprar']>=36]

final



c:\Users\Igor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning:

lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



,qtde_proposta,qtde_contato,media_tempo_proposta,media_valor_proposta,%_não_comprar,%_comprar,cliente_id
90,6.0,8.0,50.0,19604.83,4.72,95.28,473
99,6.0,7.0,37.0,22786.00,5.35,94.65,190
78,5.0,12.0,53.0,18736.20,8.36,91.64,328
25,5.0,9.0,53.0,22122.80,8.91,91.09,460
9,5.0,3.0,68.0,25220.60,9.01,90.99,106
...,...,...,...,...,...,...,...
11,1.0,12.0,49.0,14090.00,62.98,37.02,361
67,1.0,9.0,57.0,29202.00,63.32,36.68,352
83,1.0,4.0,67.0,7541.00,63.50,36.50,16
29,1.0,8.0,56.0,24387.00,63.91,36.09,395


Verificar se o que o modelo disse está coerente.

In [96]:

y_proba = logreg.predict_proba(x_test.drop('cliente_id', axis=1))[:, 1]


from sklearn.metrics import confusion_matrix

y_pred_custom = (y_proba >= 0.36).astype(int)
matrix = pd.DataFrame(confusion_matrix(y_test, y_pred_custom))

matrix.rename(columns={0:'Modelo_Disse_NaoComprou',1:'Modelo_Disse_Comprou'},inplace=True)
matrix.rename({0:'Nao_Comprou',1:'Comprou'},inplace=True)
matrix



,Modelo_Disse_NaoComprou,Modelo_Disse_Comprou
Nao_Comprou,19,25
Comprou,5,51


In [47]:
# clientes com valor medio da proposta maior e alta % de comprar:
display(final.sort_values(by=['media_valor_proposta','%_comprar'],ascending=[False,False]).head(10))

# clientes com alta % de comprar:
display(final.sort_values(by='%_comprar',ascending=False).head(10))

,qtde_proposta,qtde_contato,media_tempo_proposta,media_valor_proposta,%_não_comprar,%_comprar,cliente_id
45,1.0,8.0,43.0,39074.0,66.65,33.35,83
50,3.0,8.0,60.0,37026.0,28.95,71.05,63
93,1.0,6.0,56.0,35993.0,65.17,34.83,192
28,3.0,6.0,72.0,35788.0,27.81,72.19,374
64,2.0,5.0,29.0,35343.5,53.65,46.35,380
3,1.0,5.0,62.0,35112.0,64.53,35.47,174
47,3.0,4.0,42.0,34424.0,33.99,66.01,136
26,2.0,6.0,44.0,33062.0,49.96,50.04,348
56,2.0,10.0,42.0,32744.5,48.16,51.84,28
54,1.0,2.0,82.0,32686.0,62.19,37.81,206


,qtde_proposta,qtde_contato,media_tempo_proposta,media_valor_proposta,%_não_comprar,%_comprar,cliente_id
90,6.0,8.0,50.0,19604.83,4.72,95.28,473
99,6.0,7.0,37.0,22786.00,5.35,94.65,190
78,5.0,12.0,53.0,18736.20,8.36,91.64,328
25,5.0,9.0,53.0,22122.80,8.91,91.09,460
9,5.0,3.0,68.0,25220.60,9.01,90.99,106
86,5.0,4.0,59.0,23875.80,9.43,90.57,290
76,5.0,7.0,40.0,24026.00,10.23,89.77,264
4,5.0,4.0,47.0,20189.60,10.28,89.72,35
69,4.0,6.0,74.0,22214.75,15.35,84.65,458
46,4.0,7.0,68.0,19724.50,15.68,84.32,90


Teste de modelo para ver acurácia: GradientBoostingClassifier

In [5]:

from sklearn.ensemble import GradientBoostingClassifier
cliente_id_merge = x_test['cliente_id'].reset_index().drop('index',axis=1)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(x_train.drop('cliente_id', axis=1), y_train)

gb_proba = gb.predict_proba(x_test.drop('cliente_id', axis=1))
gb_proba1 = gb.predict_proba(x_test.drop('cliente_id', axis=1))[:, 1]


teste_gb = pd.DataFrame(x_test.drop('cliente_id', axis=1)).reset_index().drop('index',axis=1)

probs_gb = pd.DataFrame(gb_proba)

display(probs_gb)



final = pd.concat([teste_gb,probs_gb, cliente_id_merge],axis = 1)

final.rename(columns={0:'%_não_comprar', 1: '%_comprar'},inplace=True)

final

final['%_não_comprar'] = (final['%_não_comprar']*100).round(decimals=2)
final['%_comprar'] = (final['%_comprar']*100).round(decimals=2)

final = final.sort_values(by='%_comprar',ascending=False)

final = final[final['%_comprar']>=36]

display(final)

from sklearn.metrics import confusion_matrix

y_pred_custom = (gb_proba1 >= 0.32).astype(int)
matrix2 = pd.DataFrame(confusion_matrix(y_test, y_pred_custom))

matrix2.rename(columns={0:'Modelo_Disse_NaoComprou',1:'Modelo_Disse_Comprou'},inplace=True)
matrix2.rename({0:'Nao_Comprou',1:'Comprou'},inplace=True)
matrix2



,0,1
0,0.576767,0.423233
1,0.229664,0.770336
2,0.536572,0.463428
3,0.924262,0.075738
4,0.269408,0.730592
...,...,...
95,0.310566,0.689434
96,0.170436,0.829564
97,0.132208,0.867792
98,0.416695,0.583305


,qtde_proposta,qtde_contato,media_tempo_proposta,media_valor_proposta,%_não_comprar,%_comprar,cliente_id
86,5.0,4.0,59.0,23875.8,2.77,97.23,290
26,2.0,6.0,44.0,33062.0,3.21,96.79,348
47,3.0,4.0,42.0,34424.0,5.79,94.21,136
78,5.0,12.0,53.0,18736.2,6.88,93.12,328
50,3.0,8.0,60.0,37026.0,7.65,92.35,63
...,...,...,...,...,...,...,...
8,2.0,3.0,74.0,23260.5,55.38,44.62,440
81,2.0,7.0,44.0,24631.0,55.92,44.08,366
24,2.0,4.0,21.0,19991.5,56.68,43.32,239
0,1.0,8.0,46.0,17781.0,57.68,42.32,138


,Modelo_Disse_NaoComprou,Modelo_Disse_Comprou
Nao_Comprou,13,31
Comprou,5,51


Gráfico confusion matrices

In [124]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "table"}, {"type": "table"}]],
    subplot_titles=("Gradient Boosting", "Logistic Regression",)
)

fig.update_layout(height=300, width=1500, title_text="Confusion Matrices")

fig.add_trace(go.Table(header=dict(values=["",'Modelo_Disse_NaoComprou', 'Modelo_Disse_Comprou']),
                       cells=dict(values=[["Nao_comprou",'Comprou'],matrix2['Modelo_Disse_NaoComprou'],matrix2['Modelo_Disse_Comprou']])),row=1, col=1)


fig.add_trace(go.Table(header=dict(values=["",'Modelo_Disse_NaoComprou', 'Modelo_Disse_Comprou']),
                       cells=dict(values=[["Nao_comprou",'Comprou'],matrix['Modelo_Disse_NaoComprou'],matrix['Modelo_Disse_Comprou']])),row=1, col=2)

fig.update_layout(title_text="Matrix", xaxis_title="Gradient Boosting", yaxis_title="linear regression")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'cells': {'values': [['Nao_comprou', 'Comprou'], [13, 5], [31, 51]]},
              'domain': {'x': [0.0, 0.45], 'y': [0.0, 1.0]},
              'header': {'values': ['', 'Modelo_Disse_NaoComprou', 'Modelo_Disse_Comprou']},
              'type': 'table'},
             {'cells': {'values': [['Nao_comprou', 'Comprou'], [19, 5], [25, 51]]},
              'domain': {'x': [0.55, 1.0], 'y': [0.0, 1.0]},
              'header': {'values': ['', 'Modelo_Disse_NaoComprou', 'Modelo_Disse_Comprou']},
              'type': 'table'}],
    'layout': {'annotations': [{'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Gradient Boosting',
                                'x': 0.225,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'},
                               {'font': {'size': 16},
                                'showarrow': False,
                                'text': 'Logistic Regression',
                                'x': 0.775,
                                'xanchor': 'center',
                                'xref': 'paper',
                                'y': 1.0,
                                'yanchor': 'bottom',
                                'yref': 'paper'}],
               'height': 300,
               'template': '...',
               'title': {'text': 'Matrix'},
               'width': 1500,
               'xaxis': {'title': {'text': 'Gradient Boosting'}},
               'yaxis': {'title': {'text': 'linear regression'}}}
})

Teste de modelo para ver acurácia: RandomForestClassifier

In [49]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

rf.fit(x_train.drop('cliente_id', axis=1), y_train)

rf_proba = rf.predict_proba(x_test.drop('cliente_id', axis=1))[:, 1]


Verificar qual modelo melhor se alinhou ao dataset

In [86]:
from sklearn.metrics import roc_auc_score
import plotly.express as px

print(roc_auc_score(y_test, gb_proba1))
print(roc_auc_score(y_test, y_proba))
print(roc_auc_score(y_test, rf_proba))

scores = [("gradient boosting", roc_auc_score(y_test, gb_proba1)), ("logistic regression", roc_auc_score(y_test, y_proba)), ("random forest", roc_auc_score(y_test, rf_proba))]

melhor_modelo = sorted(scores, key=lambda x: x[1], reverse=True)[0]
print(f'O modelo que melhor se alinhou ao dataset foi o modelo de {melhor_modelo[0]}: {melhor_modelo[1]}')

bar = px.bar(x=['gradient boosting', 'logistic regression', 'random forest'], y=[roc_auc_score(y_test, gb_proba1), roc_auc_score(y_test, y_proba), roc_auc_score(y_test, rf_proba)], labels={'x':'Modelos', 'y':'ROC AUC Score'}, title='Comparação de Modelos - ROC AUC Score', text=[f'{roc_auc_score(y_test, gb_proba1):.4f}', f'{roc_auc_score(y_test, y_proba):.4f}', f'{roc_auc_score(y_test, rf_proba):.4f}'])
bar.write_image("model_comparison_roc_auc.png")

0.7297077922077921
0.726461038961039
0.6751217532467533
O modelo que melhor se alinhou ao dataset foi o modelo de gradient boosting: 0.7297077922077921
